In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [35]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [36]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [37]:
len(splitted_docs)

26

In [38]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding = embeddings
)

In [ ]:
### agent = tools, llm, prompt

In [39]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to find relevant data of the PDF Documents, 
        and these documents has details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    
    for doc in docs:
        context += doc.page_content + "\n\n"
    
    return context


In [40]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [41]:
System_prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the 'retriever_tool' tool for questions requiring external knowledge.
"""

In [42]:
agent = create_agent(
    model = llm,
    tools = [retriever_tool],
    system_prompt = System_prompt
)

In [47]:
query = "What is the name of Patient and their RBC count, and waht is the name of Doctors?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  patient name RBC count doctor name


In [48]:
result = response["messages"][-1].content

In [49]:
print(result)

**Patient:** Ms. Nikita Chudhary  
**RBC count:** 34.47 mill/mm³ (as reported in the Hemogram section)  
**Doctor:** Dr. Nitin Nahar
